In [14]:
import sys
import petsc4py
petsc4py.init(sys.argv)
from petsc4py import PETSc
import numpy as np

class NavierStokesSolver:
    """
    Solves the time-dependent Navier-Stokes equations in 2D.
    Translates logic from PETSc C example ex46.c.
    """
    def __init__(self, reynolds=400.0):
        self.re = reynolds
        self.dm = None
        self.ts = None
        self.appctx = {"re": reynolds}

    def create_mesh(self):
        """Creates a parallel unstructured mesh (DMPLEX)."""
        # Using simplex=False for built-in quad support to ensure 
        # compatibility without external grid generators.
        self.dm = PETSc.DMPlex().createBoxMesh(
            faces=[4, 4], 
            lower=[0, 0], 
            upper=[1, 1], 
            simplex=False, 
            interpolate=True
        )
        self.dm.setFromOptions()

    def setup_discretization(self):
        """Sets up Finite Element spaces (Q2-Q1 Taylor-Hood equivalent)."""
        dim = int(self.dm.getDimension())
        
        # Syntax: createDefault(dim, nc, isSimplex, qorder, prefix, comm)
        qorder = 3
        
        # Velocity Field (Field 0): Quadratic (Q2)
        fe_vel = PETSc.FE().createDefault(dim, dim, False, qorder, "vel_", None)
        
        # Pressure Field (Field 1): Linear (Q1)
        fe_pres = PETSc.FE().createDefault(dim, 1, False, qorder, "pres_", None)
        
        self.dm.setField(0, fe_vel)
        self.dm.setField(1, fe_pres)
        self.dm.createDS()

    def setup_problem(self):
        """
        Defines the physics residuals.
        Note: In Phase 2, you will define weak form kernels (f0, f1) here.
        """
        ds = self.dm.getDS()
        
        # Register the pointwise physics kernels for Velocity (Field 0) and Pressure (Field 1)
        ds.setResidual(0, self.f0_u, self.f1_u)
        ds.setResidual(1, self.f0_p, self.f1_p)
        
    # --- Weak Form (Pointwise) Kernels Translated from ex46.c ---
    @staticmethod
    def f0_u(dim, Nf, NfAux, uOff, uOff_x, u, u_t, u_x, aOff, aOff_x, a, a_t, a_x, t, x, numConstants, constants, f0):
        # f0_u = u_t + u \cdot \nabla u - f
        Re = 400.0
        
        # MMS1 Source terms
        fx = -2.0*t*(x[0] + x[1]) + 2.0*x[0]*x[1]**2 - 4.0*x[0]**2*x[1] - 2.0*x[0]**3 + 4.0/Re - 1.0
        fy = -2.0*t*x[0] + 2.0*x[1]**3 - 4.0*x[0]*x[1]**2 - 2.0*x[0]**2*x[1] + 4.0/Re - 1.0
        
        # Convection: u \cdot \nabla u
        # u_x is flattened: [du/dx, du/dy, dv/dx, dv/dy]
        u_conv = u[0]*u_x[0] + u[1]*u_x[1]
        v_conv = u[0]*u_x[2] + u[1]*u_x[3]
        
        f0[0] = u_t[0] + u_conv - fx
        f0[1] = u_t[1] + v_conv - fy
        
    @staticmethod
    def f1_u(dim, Nf, NfAux, uOff, uOff_x, u, u_t, u_x, aOff, aOff_x, a, a_t, a_x, t, x, numConstants, constants, f1):
        # f1_u = \nu \nabla u - p I
        Re = 400.0
        nu = 1.0 / Re
        p = u[uOff[1]] # Pressure is at the offset for Field 1
        
        f1[0] = nu * u_x[0] - p
        f1[1] = nu * u_x[1]
        f1[2] = nu * u_x[2]
        f1[3] = nu * u_x[3] - p
        
    @staticmethod
    def f0_p(dim, Nf, NfAux, uOff, uOff_x, u, u_t, u_x, aOff, aOff_x, a, a_t, a_x, t, x, numConstants, constants, f0):
        # f0_p = \nabla \cdot u
        f0[0] = u_x[0] + u_x[3]
        
    @staticmethod
    def f1_p(dim, Nf, NfAux, uOff, uOff_x, u, u_t, u_x, aOff, aOff_x, a, a_t, a_x, t, x, numConstants, constants, f1):
        # No gradient test functions for the continuity equation
        f1[0] = 0.0
        f1[1] = 0.0

    def solve(self, max_steps=5, dt=0.01):
        """Configures and runs the TS solver."""
        self.ts = PETSc.TS().create(PETSc.COMM_WORLD)
        self.ts.setDM(self.dm)
        self.ts.setProblemType(self.ts.ProblemType.NONLINEAR)
        
        # FIX: Based on TS.setIFunction docs, passing ts.computeIFunction recursively 
        # triggers Error 101. By completely omitting the global override, the TS automatically 
        # pulls the local FEM evaluation logic (DMPlexTSComputeIFunctionFEM) from the DM 
        # using the pointwise kernels we defined above.
        
        self.ts.setTimeStep(dt)
        self.ts.setMaxSteps(max_steps)
        self.ts.setExactFinalTime(PETSc.TS.ExactFinalTime.STEPOVER)
        
        # Finalize options from command line
        self.ts.setFromOptions()
        
        # Initial condition (Required for solving)
        u = self.dm.createGlobalVec()
        u.set(0.0)
        
        print(f"Starting solver infrastructure (Re={self.re})...")
        try:
            # We solve using the initial vector
            self.ts.solve(u)
            print("Solver finished successfully.")
        except PETSc.Error as e:
            # For Phase 1, we expect a 'converged' state of 0 or a SNES failure
            # if no physics are defined. Catching it here allows the script to finish.
            print(f"PETSc solver initialized. Status: {e}")
            
        return u

if __name__ == "__main__":
    try:
        solver = NavierStokesSolver()
        solver.create_mesh()
        solver.setup_discretization()
        solver.setup_problem()
        solution = solver.solve()
    except Exception as e:
        import traceback
        traceback.print_exc()

Traceback (most recent call last):
  File "/var/folders/gz/092twjls1kj29pp_6mhckn4w0000gn/T/ipykernel_3872/4160315925.py", line 139, in <module>
    solver.setup_problem()
  File "/var/folders/gz/092twjls1kj29pp_6mhckn4w0000gn/T/ipykernel_3872/4160315925.py", line 56, in setup_problem
    ds.setResidual(0, self.f0_u, self.f1_u)
AttributeError: 'petsc4py.PETSc.DS' object has no attribute 'setResidual'
